In [2]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving ecWvPsdV2FNmRZAa (4).csv to ecWvPsdV2FNmRZAa (4).csv


In [3]:
!pip install -q google-genai pydantic

import pandas as pd
import json
import time
from google.colab import files
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

notes = pd.read_csv("followup_notes.csv")

print("Total notes:", len(notes))
display(notes.head())

Total notes: 30


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [4]:

from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
from getpass import getpass
import json
import time

API_KEY = getpass("Enter Gemini API key: ")
client = genai.Client(api_key=API_KEY)
del API_KEY

MODEL_NAME = "gemini-3.8-flash"

ALLOWED = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

SYSTEM_PROMPT = (
    "You are a healthcare operations analyst. "
    "Analyse missed appointment notes carefully. "
    "Do not invent missing information."
)

def make_prompt(note_text):
    return f"""
Context: A patient missed a clinic appointment.
Role: Healthcare operations analyst.
Action: Identify the main reason for missing the appointment.

Allowed categories:
{', '.join(ALLOWED)}

Use 'other' if unclear.
Understand English, Hindi and Hinglish.
Tone: Neutral, brief and factual.

Format: Return JSON with root_cause, confidence,
summary and recommended_action.
Confidence must be 0 to 1.
Summary must be 25 words or fewer.

Patient note: {note_text}
"""

api_calls = 0

def analyse_note(note_text, version="v2"):
    global api_calls

    prompt = (
        make_prompt(note_text)
        if version == "v2"
        else "Classify this missed appointment note: " + str(note_text)
    )

    api_calls += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0,
            response_mime_type="application/json",
            response_schema=FollowupInsight,
            max_output_tokens=2000
        )
    )

    raw = response.text or ""
    return json.loads(raw), raw

print("Gemini setup complete!")


Enter Gemini API key: ··········
Gemini setup complete!


In [7]:
MODEL_NAME = "gemini-3.5-flash-lite"

first_note = notes.loc[
    notes["note_id"] == "N001", "note_text"
].iloc[0]

for attempt in range(3):
    try:
        first_result, first_raw = analyse_note(first_note)
        print("N001 result:")
        print(json.dumps(first_result, indent=2))
        break

    except Exception as e:
        if "503" not in str(e) or attempt == 2:
            raise

        print("Server busy. Retrying in 30 seconds...")
        time.sleep(30)

print("API calls:", api_calls)

N001 result:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "Patient missed the appointment because she never received a reminder message and forgot.",
  "recommended_action": "Reschedule the appointment and verify reminder contact preferences."
}
API calls: 3


In [8]:

results = []

for _, row in notes.head(10).iterrows():
    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    if pd.isna(note_text) or not str(note_text).strip():
        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    data = {}
    raw = ""

    try:
        if note_id == "N001":
            data, raw = first_result, first_raw
        else:
            data, raw = analyse_note(note_text)

        validated = FollowupInsight.model_validate(data)

        if validated.root_cause not in ALLOWED:
            raise ValueError("Invalid root cause")

        if len(validated.summary.split()) > 25:
            raise ValueError("Summary too long")

        status = "valid"

    except Exception as e:
        status = "invalid"
        print(note_id, "Error:", str(e)[:200])

    results.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw
    })

    if note_id != "N001":
        time.sleep(13)

notes_analysed = pd.DataFrame(results)

notes_analysed = notes_analysed[[
    "note_id", "appointment_id", "root_cause",
    "confidence", "summary", "recommended_action",
    "validation_status", "raw_response"
]]

display(notes_analysed[[
    "note_id", "root_cause", "validation_status"
]])

print("Rows:", len(notes_analysed))
print("Columns:", len(notes_analysed.columns))
print("API calls:", api_calls)

notes_analysed.to_csv("notes_analysed.csv", index=False)

from google.colab import files
files.download("notes_analysed.csv")


,note_id,root_cause,validation_status
0,N001,forgot_or_no_reminder,valid
1,N002,distance_or_transport,valid
2,N003,cost_or_fee,valid
3,N004,other,skipped_blank
4,N005,illness_or_emergency,valid
5,N006,work_or_schedule,valid
6,N007,forgot_or_no_reminder,valid
7,N008,distance_or_transport,valid
8,N009,forgot_or_no_reminder,valid
9,N010,work_or_schedule,valid


Rows: 10
Columns: 8
API calls: 11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

invalid_examples = [
    {
        "root_cause": "cost_or_fee",
        "confidence": 1.7,
        "summary": "Patient could not afford the fee",
        "recommended_action": "Discuss payment options"
    },
    {
        "root_cause": "work_or_schedule",
        "confidence": 0.8,
        "summary": "Patient was busy at work"
    }
]

for i, example in enumerate(invalid_examples, 1):
    try:
        FollowupInsight.model_validate(example)
        print("Example", i, "unexpectedly passed")
    except ValidationError as e:
        print("Example", i, "correctly rejected")
        print(str(e)[:350])


Example 1 correctly rejected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Example 2 correctly rejected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...tient was busy at work'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [10]:

comparison = []

for note_id in ["N007", "N009"]:
    note_text = notes.loc[
        notes["note_id"] == note_id, "note_text"
    ].iloc[0]

    for version in ["v1", "v2"]:
        try:
            data, raw = analyse_note(note_text, version)

            validated = FollowupInsight.model_validate(data)
            cause = validated.root_cause

            status = (
                "valid" if cause in ALLOWED else "invalid"
            )

        except Exception as e:
            cause = "API or validation error"
            status = "invalid"
            print(note_id, version, str(e)[:150])

        comparison.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": cause,
            "validation_status": status
        })

        time.sleep(13)

prompt_comparison = pd.DataFrame(comparison)
display(prompt_comparison)

print("Total API calls:", api_calls)


,note_id,prompt_version,root_cause,validation_status
0,N007,v1,Patient forgot the appointment because they di...,invalid
1,N007,v2,forgot_or_no_reminder,valid
2,N009,v1,Patient unreachable,invalid
3,N009,v2,forgot_or_no_reminder,valid


Total API calls: 15


In [11]:

from google.colab import files

print("Upload appointments_clean.csv")
uploaded = files.upload()

appointments_clean = pd.read_csv("appointments_clean.csv")

merged = notes_analysed.merge(
    appointments_clean,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

print("Merged rows:", len(merged))

print("\nRoot cause vs reminder sent:")
display(pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
))

print("\nRoot cause counts:")
display(notes_analysed["root_cause"].value_counts())

print("\nValidation results:")
display(notes_analysed["validation_status"].value_counts())

print("\nOverall no-show percentage:")
print(round(
    appointments_clean["no_show"].mean() * 100, 2
))

print("\nTotal API calls:", api_calls)


Upload appointments_clean.csv


Saving appointments_clean.csv to appointments_clean.csv
Merged rows: 10

Root cause vs reminder sent:


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,1
illness_or_emergency,0,1
other,1,0
work_or_schedule,1,1



Root cause counts:


,count
root_cause,
forgot_or_no_reminder,3
distance_or_transport,2
work_or_schedule,2
cost_or_fee,1
other,1
illness_or_emergency,1



Validation results:


,count
validation_status,
valid,9
skipped_blank,1



Overall no-show percentage:
22.02

Total API calls: 15
